# onefig examples

A tour of the onefig API: typed config schemas, YAML loading, CLI overrides, validation, freezing, and display.

Install: `pip install -e .` from the repo root.

In [ ]:
from __future__ import annotations

import tempfile
from argparse import Namespace
from pathlib import Path

from pydantic import ValidationError

from onefig import ConfigModel
from onefig.model import FrozenConfigError

## 1. Define a schema

A onefig config is a Pydantic v2 model. Subclass `ConfigModel` and nest as deeply as you like.

By default: types are validated on load *and* on every assignment, and unknown fields raise (`extra="forbid"`).

In [2]:
class OptimizerCfg(ConfigModel):
    name: str = "adamw"
    lr: float = 1e-4
    weight_decay: float = 0.0


class ModelCfg(ConfigModel):
    name: str
    hidden_size: int = 768
    num_layers: int = 12


class TrainCfg(ConfigModel):
    run_name: str
    epochs: int = 10
    model: ModelCfg
    optimizer: OptimizerCfg = OptimizerCfg()

## 2. Load from YAML

`Cfg.load(path_or_name)` parses YAML via OmegaConf, resolves any `${...}` interpolations, then validates into the Pydantic model.

In [3]:
yaml_text = """
run_name: bert-base-experiment
epochs: 5
model:
  name: bert-base-uncased
  hidden_size: 768
  num_layers: 12
optimizer:
  name: adamw
  lr: 0.0005
  weight_decay: 0.01
"""

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "train.yaml"
    path.write_text(yaml_text)
    cfg = TrainCfg.load(path)

cfg

TrainCfg(run_name='bert-base-experiment', epochs=5, model=ModelCfg(name='bert-base-uncased', hidden_size=768, num_layers=12), optimizer=OptimizerCfg(name='adamw', lr=0.0005, weight_decay=0.01))

## 3. Validation

Type mismatches and unknown fields raise `ValidationError` at load time, not deep in your training loop.

In [4]:
bad_yaml = """
run_name: oops
epochs: not-an-int
model:
  name: bert
"""

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "bad.yaml"
    path.write_text(bad_yaml)
    try:
        TrainCfg.load(path)
    except ValidationError as e:
        print(e)

1 validation error for TrainCfg
epochs
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not-an-int', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing


In [5]:
extra_yaml = """
run_name: oops
model:
  name: bert
typo_field: 42
"""

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "extra.yaml"
    path.write_text(extra_yaml)
    try:
        TrainCfg.load(path)
    except ValidationError as e:
        print(e)

1 validation error for TrainCfg
typo_field
  Extra inputs are not permitted [type=extra_forbidden, input_value=42, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/extra_forbidden


## 4. CLI-style overrides

`update_from_args` accepts an `argparse.Namespace` or a plain dict, and supports two key styles:

- **Leaf key** (e.g. `lr`) — convenient when the key is unique within the config.
- **Full dotted path** (e.g. `optimizer.lr`) — always works, and required when a leaf key is ambiguous.

`None` values are skipped by default, so you can wire up `argparse` with `default=None` and only override what the user actually passed.

In [ ]:
cfg = TrainCfg(run_name="demo", model=ModelCfg(name="bert-base"))

args = Namespace(lr=1e-3, epochs=20, weight_decay=None)
cfg.update_from_args(args)

print("lr           :", cfg.optimizer.lr)
print("epochs       :", cfg.epochs)
print("weight_decay :", cfg.optimizer.weight_decay)
print("  (weight_decay unchanged because args.weight_decay was None)")

In [7]:
cfg.update_from_args({"model.hidden_size": 1024, "optimizer.lr": 5e-4})
print(cfg.model.hidden_size, cfg.optimizer.lr)

1024 0.0005


### Ambiguous leaf keys

If a leaf key (e.g. `lr`) appears in more than one place, onefig refuses to guess and tells you to disambiguate.

In [8]:
class GenA(ConfigModel):
    lr: float = 1e-4


class GenB(ConfigModel):
    lr: float = 1e-3


class TwoOpt(ConfigModel):
    a: GenA = GenA()
    b: GenB = GenB()


two = TwoOpt()
try:
    two.update_from_args({"lr": 0.5})
except ValueError as e:
    print(e)

# Disambiguate with a dotted path:
two.update_from_args({"a.lr": 0.5})
print("a.lr =", two.a.lr, "| b.lr =", two.b.lr)

Ambiguous override key 'lr': matches a.lr, b.lr. Use the full dotted path instead.
a.lr = 0.5 | b.lr = 0.001


## 5. Interpolation

OmegaConf's `${other.key}` references are resolved at load time, so you can DRY up shared paths and names.

In [9]:
interp_yaml = """
run_name: ${model.name}-ep${epochs}
epochs: 3
model:
  name: tiny-bert
"""

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "interp.yaml"
    path.write_text(interp_yaml)
    interp_cfg = TrainCfg.load(path)

interp_cfg.run_name

'tiny-bert-ep3'

## 6. Freezing

Once you've finished resolving CLI args, freeze the config so accidental mutation downstream is impossible. Freeze is recursive — nested `ConfigModel`s are frozen too.

In [10]:
cfg.freeze()
print("is_frozen:", cfg.is_frozen)

try:
    cfg.epochs = 99
except FrozenConfigError as e:
    print("top-level mutation blocked:", e)

try:
    cfg.optimizer.lr = 1.0
except FrozenConfigError as e:
    print("nested mutation blocked  :", e)

is_frozen: True
top-level mutation blocked: Cannot set 'epochs': config is frozen. Create a new instance via .from_dict() or .model_copy().
nested mutation blocked  : Cannot set 'lr': config is frozen. Create a new instance via .from_dict() or .model_copy().


## 7. Flatten and display

`to_flat_dict()` produces the dotted-key shape that W&B / MLflow / TensorBoard hyperparameter loggers want.

`display()` prints an ASCII tree of the resolved config. If you need the string instead (e.g. to log it), call `onefig.format_tree(cfg.to_dict())`.

In [11]:
cfg.to_flat_dict()

{'run_name': 'demo',
 'epochs': 20,
 'model.name': 'bert-base',
 'model.hidden_size': 1024,
 'model.num_layers': 12,
 'optimizer.name': 'adamw',
 'optimizer.lr': 0.0005,
 'optimizer.weight_decay': 0.0}

In [ ]:
cfg.display(name="TrainCfg")

## 8. Name-or-path lookup

`load("my_config")` (no extension, no path) recursively searches `search_root` (cwd by default) for `my_config.yaml` or `my_config.yml`. Convenient when scripts are run from anywhere in the project.

In [13]:
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    nested = root / "configs" / "experiments"
    nested.mkdir(parents=True)
    (nested / "baseline.yaml").write_text(
        "run_name: baseline\nmodel:\n  name: bert-base\n"
    )

    cfg_by_name = TrainCfg.load("baseline", search_root=root)
    print(cfg_by_name.run_name, "|", cfg_by_name.model.name)

baseline | bert-base


## Recap

| Operation                | API                                  |
|--------------------------|--------------------------------------|
| Load + validate          | `MyCfg.load(name_or_path)`           |
| From dict                | `MyCfg.from_dict({...})`             |
| CLI overrides            | `cfg.update_from_args(args)`         |
| Freeze (recursive)       | `cfg.freeze()`                       |
| Nested dict              | `cfg.to_dict()`                      |
| Flat dotted dict         | `cfg.to_flat_dict()`                 |
| Print ASCII tree         | `cfg.display()`                      |
| ASCII tree as string     | `onefig.format_tree(cfg.to_dict())`  |